# Selected ensembles: supplementary chronological test assessment

This executed reading notebook loads saved, verified assessment artifacts. It performs no model fitting, prediction or selection. Two ensembles were selected using January–May validation: the best learned stack and the best equal-weight mean. Three standalone members provide context. **The 14-variant development suite has not been rescored in full on the test window.**

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

root = next(candidate for parent in (Path.cwd(), *Path.cwd().parents)
            for candidate in (parent, parent / 'Phase_3')
            if (candidate / 'outputs/regression_stacking_holdout').is_dir())
out = root / 'outputs/regression_stacking_holdout'
verification = json.loads((out / 'verification.json').read_text())
assert verification['passed'] is True
frozen = json.loads((out / 'frozen_selection.json').read_text())
tables = out / 'tables'
comparison = pd.read_csv(tables / 'test_comparison.csv')
monthly = pd.read_csv(tables / 'test_by_month.csv')
weights = pd.read_csv(tables / 'frozen_weights.csv')
audit = pd.read_csv(tables / 'meta_time_audit.csv')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')
assert frozen['selected'] == {'stack': 'stack_two_fixed', 'mean': 'mean_three_fixed'}
print('Selected from saved validation, before test prediction:', frozen['selected'])
print('Test orders:', verification['test_n'])


Selected from saved validation, before test prediction: {'stack': 'stack_two_fixed', 'mean': 'mean_three_fixed'}
Test orders: 18603


## Validation and test have different roles

Validation supports development and selection. The CV column is the unweighted mean of five January–May monthly MAEs, using labels observed before 1 June. The June–August test column pools 18,603 completed-delivery orders after configurations and weights have been frozen. Orders are retained when their delivery occurs after August.

The periods, training histories, label maturity and averaging rules differ, so scores need not match. May validation excludes outcomes not yet available at the selection cutoff. This prevents unavailable labels from influencing development, but creates incomplete maturity in that month. A lower later score does not isolate an improvement caused by stacking.

In [2]:
order = ['stack_two_fixed', 'mean_three_fixed', 'ridge_fixed', 'forest_fixed',
         'hgb63_phase2', 'HGB_risk_current_main']
comparison = comparison.set_index('candidate').loc[order].reset_index()
assert comparison['n'].eq(18603).all()
display(comparison[['candidate', 'cv_mae', 'mae', 'rmse', 'r2', 'within3', 'n', 'assessment']].rename(
    columns={'cv_mae': 'CV mean monthly MAE (days)', 'mae': 'Test pooled MAE (days)',
             'rmse': 'Test pooled RMSE (days)'}))


,candidate,CV mean monthly MAE (days),Test pooled MAE (days),Test pooled RMSE (days),r2,within3,n,assessment
0,stack_two_fixed,5.4181,4.3875,5.8603,-0.0135,0.4185,18603,new supplementary historical assessment
1,mean_three_fixed,5.3700,3.9275,5.4741,0.1157,0.4847,18603,new supplementary historical assessment
2,ridge_fixed,5.4586,4.4802,5.9667,-0.0506,0.4114,18603,new supplementary historical assessment
3,forest_fixed,5.4871,4.4352,5.8635,-0.0146,0.4039,18603,new supplementary historical assessment
4,hgb63_phase2,5.5184,3.4217,5.1963,0.2031,0.5773,18603,new supplementary historical assessment
5,HGB_risk_current_main,5.2595,3.4343,5.0925,0.2347,0.5593,18603,saved original Phase 3 assessment; not refitted


`hgb63_phase2` is the migrated 63-leaf Phase 2 HGB member. `HGB_risk_current_main` is the different 15-leaf main Phase 3 model and uses its saved test result without refitting. Their feature representations and specifications differ as well; this table does not isolate learner complexity.

The lowest test row is a descriptive observation. **The test table does not reopen model selection or deployment.** The existing main-model decision remains unchanged.

## Monthly test results

Each of the five newly assessed predictors appears for all three purchase months. The main-model row above comes from saved evidence and is not newly scored in this notebook.

In [3]:
display(monthly[['candidate', 'month', 'n', 'mae', 'rmse', 'bias', 'within3']])
display(monthly.pivot(index='candidate', columns='month', values='mae').reindex(order[:-1]))


,candidate,month,n,mae,rmse,bias,within3
0,stack_two_fixed,2018-06,6096,5.1145,6.7075,3.6231,0.3286
1,stack_two_fixed,2018-07,6156,4.4773,6.0274,2.3249,0.4118
2,stack_two_fixed,2018-08,6351,3.6025,4.7115,1.8094,0.5114
3,mean_three_fixed,2018-06,6096,4.5249,6.2120,2.8462,0.3930
4,mean_three_fixed,2018-07,6156,4.0864,5.7199,1.7423,0.4727
5,mean_three_fixed,2018-08,6351,3.2001,4.3615,1.2463,0.5842
6,ridge_fixed,2018-06,6096,5.1786,6.8178,3.6102,0.3322
7,ridge_fixed,2018-07,6156,4.5731,6.1295,2.2013,0.4003
8,ridge_fixed,2018-08,6351,3.7196,4.8216,1.5690,0.4983
9,forest_fixed,2018-06,6096,5.1242,6.6817,3.6448,0.3141


month,2018-06,2018-07,2018-08
candidate,,,
stack_two_fixed,5.1145,4.4773,3.6025
mean_three_fixed,4.5249,4.0864,3.2001
ridge_fixed,5.1786,4.5731,3.7196
forest_fixed,5.1242,4.5083,3.7031
hgb63_phase2,3.6589,3.6336,2.9886


## Final weights and time boundaries

Final training requires `purchase >= 2017-01-01`, `purchase < 2018-06-01 - 39 days` and `delivery < 2018-06-01`. Three forward OOF windows within that partition supply the stack's meta-training rows: February, March and a partial April. Each inner fit uses purchases before its own cutoff minus 39 days and labels observed before its cutoff. Meta-labels must be observed before 1 June.

The stack's MAE-optimal nonnegative weights sum to one, with no intercept. The mean uses fixed one-third weights. Every final specification and weight was frozen before any test prediction; the meta-fit score is a training diagnostic.

In [4]:
display(weights)
display(pd.DataFrame({'member': frozen['mean_members'], 'weight': frozen['mean_weights']}))
display(audit)
print('OOF meta rows:', frozen['meta_rows'])
print('OOF meta-fit MAE (training diagnostic):', f"{frozen['meta_fit_mae']:.4f}")
print('Weights frozen at:', frozen['frozen_at'])
for values in (frozen['stack_weights'], frozen['mean_weights']):
    assert min(values) >= 0
    np.testing.assert_allclose(sum(values), 1, atol=1e-12)


,candidate,member,weight
0,stack_two_fixed,ridge_fixed,0.6282
1,stack_two_fixed,forest_fixed,0.3718


,member,weight
0,ridge_fixed,0.3333
1,forest_fixed,0.3333
2,hgb,0.3333


,month,cutoff,gap_days,train_n,score_n,train_ids_sha256,score_ids_sha256,latest_train_purchase,latest_train_label,latest_score_label,meta_label_cutoff,overlap,future_training_labels
0,2018-02,2018-02-01 00:00:00,39,42450,6551,e91e4d5d3dbbfe978a1ec09016c313feea92ba9b9c0a8c6b967859af595b5667,ee5333ed24249e8b4c0ea1ea5b91922f807a6131f9242b10503574081d7682cf,2017-12-23 23:49:03,2018-01-31 23:45:08,2018-05-30 23:50:44,2018-06-01 00:00:00,0,0
1,2018-03,2018-03-01 00:00:00,39,47940,6996,7b47e4d9a42f41f88a58504691ebf73259c7d98b242be605d5b616819febd0e9,9b36752b0b4c653f8a5ba496a70daea381069f621dabad0ae0bb8fea131d2ddc,2018-01-20 23:58:24,2018-02-28 23:48:36,2018-05-27 10:12:33,2018-06-01 00:00:00,0,0
2,2018-04,2018-04-01 00:00:00,39,54882,4875,6ee3e4090d9883a31383320c318436fca8b3b3955a756609d8874efb18720dd0,b0fb3ec3562cf2549b14919dafd711794031c009028b0cf60b6f04cac45f036c,2018-02-20 23:46:53,2018-03-31 13:51:44,2018-05-30 21:58:51,2018-06-01 00:00:00,0,0


OOF meta rows: 18422
OOF meta-fit MAE (training diagnostic): 6.2577
Weights frozen at: 2026-10-30T11:15:04.444838+00:00


## Interpretation limits and verification

These orders were exposed during Phase 2 and the earlier Phase 3 assessment. This is a supplementary historical assessment, not a new independent unseen holdout. It is conditional on completed deliveries and cannot establish performance for every placed order. The previous main-model results, test lock, development outputs and notebooks remain preserved.

The verifier replays saved ensemble predictions and weights and checks time boundaries and unchanged original evidence. This reading notebook only displays the resulting artifacts.

In [5]:
display(verification)

{'passed': True,
 'verified_at': '2026-10-30T11:19:38.168494+00:00',
 'replayed_selected_ensembles': 2,
 'test_n': 18603,
 'strict_training_and_meta_label_boundaries': True,
 'frozen_weights_and_prediction_replay': True,
 'original_evidence_unchanged': True,
 'historical_test_exposure_disclosed': True}